In [ ]:
# Silver Transform — Notebook de respaldo
# Crea users_hist y posts_hist en Iceberg (Silver)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = (
    SparkSession.builder
    .appName("silver_notebook")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "software.amazon.awssdk:bundle:2.24.8,"
        "software.amazon.awssdk:url-connection-client:2.24.8,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
        "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    .config("spark.sql.defaultCatalog", "nessie")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1")
    .config("spark.sql.catalog.nessie.ref", "main")
    .config("spark.sql.catalog.nessie.authentication.type", "NONE")
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/")
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.region", "us-east-1")
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin")
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)
print(f"Spark: {spark.version}")

In [ ]:
# Silver: users_hist (MERGE)
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")

df_u2020 = spark.read.parquet("s3a://bronze/users/2020/users_2020.parquet")
df_u2021 = spark.read.parquet("s3a://bronze/users/2021/users_2021.parquet")
df_users = df_u2020.unionByName(df_u2021, allowMissingColumns=True)
df_users = df_users.withColumn("fecha_cargue", current_timestamp())
print(f"Users combinados: {df_users.count()}")

try:
    spark.sql("SELECT 1 FROM nessie.silver.users_hist LIMIT 1")
    df_users.createOrReplaceTempView("users_in")
    spark.sql('''
        MERGE INTO nessie.silver.users_hist AS t
        USING users_in AS s ON t.Id = s.Id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    ''')
    print("MERGE users_hist completado")
except:
    df_users.writeTo("nessie.silver.users_hist").create()
    print("Tabla users_hist creada")

In [ ]:
# Silver: posts_hist (MERGE)
df_p2020 = spark.read.parquet("s3a://bronze/posts/2020/posts_2020.parquet")
df_p2021 = spark.read.parquet("s3a://bronze/posts/2021/posts_2021.parquet")
df_posts = df_p2020.unionByName(df_p2021, allowMissingColumns=True)
df_posts = df_posts.withColumn("fecha_cargue", current_timestamp())
print(f"Posts combinados: {df_posts.count()}")

try:
    spark.sql("SELECT 1 FROM nessie.silver.posts_hist LIMIT 1")
    df_posts.createOrReplaceTempView("posts_in")
    spark.sql('''
        MERGE INTO nessie.silver.posts_hist AS t
        USING posts_in AS s ON t.Id = s.Id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    ''')
    print("MERGE posts_hist completado")
except:
    df_posts.writeTo("nessie.silver.posts_hist").create()
    print("Tabla posts_hist creada")

In [ ]:
# Verificar
spark.sql("SELECT COUNT(*) as total FROM nessie.silver.users_hist").show()
spark.sql("SELECT COUNT(*) as total FROM nessie.silver.posts_hist").show()
spark.stop()